# 05 - Data Preprocessing Pipeline

This notebook builds a reusable Scikit-learn preprocessing pipeline for the fraud detection system. The pipeline standardizes selected numerical features while leaving the remaining features unchanged, ensuring consistent preprocessing during training, evaluation, and deployment.

In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load Dataset

Load the cleaned dataset and recreate the engineered features developed during the feature engineering phase.

In [2]:
DATA_PATH = Path("../data/raw/creditcard.csv")

df = pd.read_csv(DATA_PATH)

df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns="Class")
y = df["Class"]

## Recreate Engineered Features

Rebuild the engineered features so that the preprocessing pipeline operates on the complete feature set.

In [3]:
import numpy as np

X["TransactionHour"] = (X["Time"] // 3600).astype(int)

X["LogAmount"] = np.log1p(X["Amount"])

threshold = X["Amount"].quantile(0.95)

X["HighValueTransaction"] = (
    X["Amount"] > threshold
).astype(int)

print(f"Total Features: {X.shape[1]}")

Total Features: 33


## Identify Features for Scaling

Only continuous numerical features require standardization. PCA-transformed features are already standardized and therefore remain unchanged.

In [4]:
features_to_scale = [
    "Time",
    "Amount",
    "TransactionHour",
    "LogAmount"
]

print(features_to_scale)

['Time', 'Amount', 'TransactionHour', 'LogAmount']


## Build the Preprocessing Pipeline

Construct a reusable preprocessing pipeline using a `ColumnTransformer`. The selected continuous features will be standardized, while all remaining features will pass through unchanged.

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "scaler",
            StandardScaler(),
            features_to_scale
        )
    ],
    remainder="passthrough"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

## Pipeline Summary

Display the constructed preprocessing pipeline to verify its configuration before it is used during model training.

In [6]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

## Pipeline Components

Review the individual components that make up the preprocessing pipeline.

In [7]:
pipeline.named_steps

{'preprocessor': ColumnTransformer(remainder='passthrough',
                   transformers=[('scaler', StandardScaler(),
                                  ['Time', 'Amount', 'TransactionHour',
                                   'LogAmount'])])}

## Preprocessing Summary

The preprocessing pipeline has been successfully created.

- StandardScaler will standardize continuous features.
- PCA-transformed features remain unchanged.
- Binary engineered features pass through unchanged.
- The pipeline is reusable across training, validation, testing, and inference.
- The pipeline has **not been fitted** yet, preventing data leakage.

## Validate Pipeline Structure

Inspect the preprocessing pipeline to verify that all components have been configured correctly before it is fitted during model training.

In [8]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

## Inspect Pipeline Steps

Review the individual steps contained within the preprocessing pipeline.

In [9]:
pipeline.steps

[('preprocessor',
  ColumnTransformer(remainder='passthrough',
                    transformers=[('scaler', StandardScaler(),
                                   ['Time', 'Amount', 'TransactionHour',
                                    'LogAmount'])]))]

## Inspect Column Transformer

Verify that the preprocessing pipeline contains the expected transformer configuration.

In [10]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.

## Verify Selected Features

Ensure that the intended continuous features have been selected for standardization.

In [11]:
print("Features selected for scaling:")

for feature in features_to_scale:
    print(f"- {feature}")

Features selected for scaling:
- Time
- Amount
- TransactionHour
- LogAmount


## Verify Pipeline Configuration

Display key configuration details for the preprocessing pipeline.

In [12]:
print("Pipeline Type:", type(pipeline).__name__)

print("Preprocessor Type:", type(preprocessor).__name__)

print("Transformer Names:")

for name, transformer, columns in preprocessor.transformers:
    print(f"\n{name}")
    print("Transformer :", type(transformer).__name__)
    print("Columns     :", columns)

Pipeline Type: Pipeline
Preprocessor Type: ColumnTransformer
Transformer Names:

scaler
Transformer : StandardScaler
Columns     : ['Time', 'Amount', 'TransactionHour', 'LogAmount']


## Validation Summary

The preprocessing pipeline has been successfully validated.

Validation confirms:

- The pipeline has been created successfully.
- The `ColumnTransformer` contains the expected transformer.
- The correct continuous features have been selected for scaling.
- No fitting has been performed, preventing data leakage.
- The pipeline is ready to be fitted on the training data during the model development phase.

## Preprocessing Workflow

The preprocessing pipeline has been designed to ensure consistent feature transformation throughout the machine learning lifecycle.

The workflow consists of the following stages:

1. Load the cleaned dataset.
2. Recreate engineered features.
3. Identify continuous features requiring scaling.
4. Build a reusable preprocessing pipeline.
5. Validate the pipeline configuration.
6. Fit the pipeline only on the training data during model development.
7. Apply the fitted pipeline consistently to validation, testing, and inference data.

## Preprocessing Decisions

The following design decisions were adopted:

- Continuous numerical features are standardized using `StandardScaler`.
- PCA-transformed features remain unchanged because they are already standardized.
- Binary engineered features are passed through without modification.
- A `ColumnTransformer` is used to selectively transform only the required columns.
- A Scikit-learn `Pipeline` encapsulates the preprocessing workflow for reusability.
- Pipeline fitting is intentionally deferred until after the train-test split to prevent data leakage.

## Production Readiness

The preprocessing pipeline has been designed to support production deployment.

Key characteristics include:

- Reusable preprocessing logic.
- Consistent feature transformations.
- Prevention of data leakage.
- Compatibility with model persistence using `joblib`.
- Seamless integration into inference and deployment workflows.

## Phase Summary

Phase 2.6 successfully established a reusable preprocessing framework for the fraud detection system.

Achievements:

- Built a reusable Scikit-learn preprocessing pipeline.
- Configured selective feature standardization using `ColumnTransformer`.
- Validated the preprocessing configuration.
- Documented preprocessing design decisions and workflow.

The project is now ready to proceed to dataset splitting and model development.

## Serialize Preprocessing Pipeline

In [13]:
import joblib
from pathlib import Path

# ----------------------------------------------------------
# Artifacts Directory
# ----------------------------------------------------------

artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Save Preprocessing Pipeline
# ----------------------------------------------------------

pipeline_path = artifacts_dir / "preprocessing_pipeline.pkl"

joblib.dump(
    pipeline,
    pipeline_path
)

print(f"Pipeline saved successfully:\n{pipeline_path}")

Pipeline saved successfully:
..\artifacts\preprocessing_pipeline.pkl


In [15]:
import joblib

loaded_pipeline = joblib.load("../artifacts/preprocessing_pipeline.pkl")

print(type(loaded_pipeline))
print(loaded_pipeline)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaler', StandardScaler(),
                                                  ['Time', 'Amount',
                                                   'TransactionHour',
                                                   'LogAmount'])]))])


In [14]:
loaded_pipeline = joblib.load(pipeline_path)

print(type(loaded_pipeline))

<class 'sklearn.pipeline.Pipeline'>
